### Notebook to illustrate how to read and visualize camera geometries

In [ ]:
import geopandas as gpd
import sealoc.models as models

from pathlib import Path
from loguru import logger
from sealoc.dal import create_data_access_layer, DataAccessLayer

In [ ]:
dal: DataAccessLayer = create_data_access_layer(
    database_url="sqlite:////data/sealoc/sealoc.db",
)

with dal.session() as repos:
    pose_ids: list[int] = repos.poses.get_all_ids()
    camera_poses: list[models.CameraPose] = repos.poses.get_all_by_ids(pose_ids)

    footprint_ids: list[int] = repos.footprints.get_all_ids()
    camera_footprints: list[models.CameraFootprint] = repos.footprints.get_all_by_ids(
        footprint_ids
    )

logger.info(f"Camera poses:       {len(camera_poses)}")
logger.info(f"Camera footprints:  {len(camera_footprints)}")

In [ ]:
def export_camera_poses(
    camera_poses: list[models.CameraPose], export_path: Path
) -> None:
    """Export camera poses to a GeoJSON file."""
    reference_pose: models.CameraPose = camera_poses[0]
    rows: list[dict] = [
        {"camera_id": pose.camera_id, "geometry": pose.shape}
        for pose in camera_poses
    ]
    frame: gpd.GeoDataFrame = gpd.GeoDataFrame(rows, crs=reference_pose.crs)
    frame.to_file(export_path, driver="GeoJSON")


def export_camera_footprints(
    camera_footprints: list[models.CameraFootprint], export_path: Path
) -> None:
    """Export camera footprints to a GeoJSON file."""
    reference_footprint: models.CameraFootprint = camera_footprints[0]
    rows: list[dict] = [
        {"camera_id": footprint.camera_id, "geometry": footprint.shape}
        for footprint in camera_footprints
    ]
    frame: gpd.GeoDataFrame = gpd.GeoDataFrame(rows, crs=reference_footprint.crs)
    frame.to_file(export_path, driver="GeoJSON")


export_dir: Path = Path("/data/sealoc")
export_camera_poses(camera_poses, export_dir / "camera_poses.geojson")
export_camera_footprints(camera_footprints, export_dir / "camera_footprints.geojson")